# Clase 222 — Surprise + Implicit + LightFM: benchmark comparativo

Mismo dataset sintético, las 3 librerías. NDCG@10 + tiempo de entrenamiento.

Requiere: `pip install scikit-surprise implicit lightfm scipy`.

In [ ]:
import os; os.environ['OPENBLAS_NUM_THREADS'] = '1'   # evitar warning de implicit
import numpy as np, pandas as pd, time
from scipy.sparse import csr_matrix

rng = np.random.default_rng(42)
n_users, n_items = 500, 300

# Generar ratings sintéticos
P_true = rng.normal(0, 1, (n_users, 8))
Q_true = rng.normal(0, 1, (n_items, 8))
scores_true = P_true @ Q_true.T
noise = rng.normal(0, 0.3, scores_true.shape)
interact_prob = 1 / (1 + np.exp(-(scores_true + noise)))
R = (rng.random(scores_true.shape) < 0.15 * interact_prob).astype(float)
ratings = np.where(R > 0, rng.integers(3, 6, R.shape), 0).astype(int)

df = pd.DataFrame([
    {'user_id': u, 'item_id': i, 'rating': int(ratings[u, i])}
    for u in range(n_users) for i in range(n_items) if ratings[u, i] > 0
])
print(f'dataset: {len(df):,} ratings')

## 1. Surprise SVD (explicit feedback)

In [ ]:
try:
    from surprise import Dataset, Reader, SVD
    from surprise.model_selection import train_test_split as surprise_split
    from surprise.accuracy import rmse

    reader = Reader(rating_scale=(1, 5))
    sd = Dataset.load_from_df(df[['user_id', 'item_id', 'rating']], reader)
    trainset, testset = surprise_split(sd, test_size=0.2, random_state=42)

    t0 = time.perf_counter()
    algo = SVD(n_factors=50, n_epochs=20, random_state=42)
    algo.fit(trainset)
    train_t = time.perf_counter() - t0

    preds = algo.test(testset)
    surprise_rmse = rmse(preds, verbose=False)
    print(f'Surprise SVD: RMSE={surprise_rmse:.4f}, train={train_t:.2f}s')
except ImportError:
    print('pip install scikit-surprise')

## 2. Implicit ALS

In [ ]:
# Split temporal-ish (random aquí)
n = len(df)
idx = rng.permutation(n)
test_idx = set(idx[:n // 5].tolist())
train_mask = np.array([i not in test_idx for i in range(n)])
df_train, df_test = df.iloc[train_mask], df.iloc[~train_mask]

R_train = csr_matrix(
    (df_train.rating, (df_train.user_id, df_train.item_id)),
    shape=(n_users, n_items),
)
R_test = csr_matrix(
    (df_test.rating, (df_test.user_id, df_test.item_id)),
    shape=(n_users, n_items),
)

try:
    import implicit
    t0 = time.perf_counter()
    model_als = implicit.als.AlternatingLeastSquares(factors=64, regularization=0.05, iterations=20, alpha=40, random_state=42)
    model_als.fit(R_train, show_progress=False)
    als_t = time.perf_counter() - t0
    print(f'Implicit ALS train: {als_t:.2f}s')

    t0 = time.perf_counter()
    model_bpr = implicit.bpr.BayesianPersonalizedRanking(factors=64, learning_rate=0.05, iterations=50, random_state=42)
    model_bpr.fit(R_train, show_progress=False)
    bpr_t = time.perf_counter() - t0
    print(f'Implicit BPR train: {bpr_t:.2f}s')
except ImportError:
    print('pip install implicit')

## 3. LightFM hybrid

In [ ]:
try:
    from lightfm import LightFM
    from lightfm.evaluation import precision_at_k as lfm_pak

    # Features sintéticos (géneros) — usar identity por default + 5 features extras
    item_feats = csr_matrix(np.hstack([
        np.eye(n_items),
        rng.binomial(1, 0.3, (n_items, 5)).astype(float),
    ]))

    t0 = time.perf_counter()
    model_lfm = LightFM(loss='warp', no_components=32, random_state=42)
    model_lfm.fit(R_train, item_features=item_feats, epochs=20, num_threads=2)
    lfm_t = time.perf_counter() - t0
    print(f'LightFM WARP train: {lfm_t:.2f}s')
except ImportError:
    print('pip install lightfm')

## 4. Evaluación uniforme: NDCG@10

In [ ]:
def ndcg_at_k(rel, k):
    rel_k = rel[:k]
    dcg = (rel_k / np.log2(np.arange(2, k + 2))).sum()
    ideal = np.sort(rel)[::-1][:k]
    idcg = (ideal / np.log2(np.arange(2, k + 2))).sum()
    return dcg / idcg if idcg > 0 else 0

def eval_implicit(model, R_train, R_test, k=10):
    R_test_dense = R_test.toarray()
    ndcgs = []
    for u in range(n_users):
        if R_test_dense[u].sum() == 0: continue
        ids, _ = model.recommend(u, R_train[u], N=k, filter_already_liked_items=True)
        rel = (R_test_dense[u, ids] > 0).astype(float)
        ndcgs.append(ndcg_at_k(rel, k))
    return float(np.mean(ndcgs))

results = []
try:
    results.append({'model': 'Implicit ALS', 'NDCG@10': eval_implicit(model_als, R_train, R_test), 'train_s': als_t})
    results.append({'model': 'Implicit BPR', 'NDCG@10': eval_implicit(model_bpr, R_train, R_test), 'train_s': bpr_t})
except NameError: pass

try:
    p_lfm = lfm_pak(model_lfm, R_test, train_interactions=R_train,
                    item_features=item_feats, k=10, num_threads=2).mean()
    results.append({'model': 'LightFM WARP', 'NDCG@10': float(p_lfm), 'train_s': lfm_t,
                    'note': '(precision@10 — proxy)'})
except NameError: pass

print(pd.DataFrame(results).round(4).to_string(index=False))

## 5. Decision matrix

In [ ]:
decision = pd.DataFrame([
    {'caso': 'aprender ABC / didáctico',           'recomendación': 'Surprise'},
    {'caso': 'producción CF, 10M-1B interactions', 'recomendación': 'Implicit ALS/BPR'},
    {'caso': 'hybrid con metadata rica',           'recomendación': 'LightFM'},
    {'caso': 'deep RS, escala TB, features ricos', 'recomendación': 'TF Recommenders / Spotlight'},
    {'caso': 'escala distribuida en Spark',        'recomendación': 'pyspark.ml.recommendation.ALS'},
    {'caso': 'serving embeddings <10ms p99',       'recomendación': 'FAISS (in-process) / Milvus (server)'},
])
print(decision.to_string(index=False))

## Ejercicio guiado

1. Replicá sobre MovieLens 1M real. Compará Surprise SVD vs Implicit ALS vs LightFM en NDCG@10 + tiempo.
2. Servir el mejor modelo: FastAPI (Clase 199) + FAISS index sobre `item_factors`. Medir p99 latency.
3. Probar TF Recommenders con tutorial oficial MovieLens — comparar effort vs Implicit.
4. Bonus: levantar `Milvus` en docker-compose. Indexar embeddings allí. Comparar con FAISS local.
5. Cost analysis: cuánto cuesta entrenar + servir 100M interactions con cada librería (cloud bill).

## Conclusiones

- 2026 default Python: **Implicit** para CF, **LightFM** si tenés features.
- TF Recommenders / two-tower para deep RS serio.
- Surprise solo educativo.
- Serving: FAISS para in-process, Milvus/Pinecone para gestionado.
- **Fin de Parte 6**: CF + content + hybrid + métricas + cold-start + librerías = stack completo de RS.

## ✅ Soluciones de los ejercicios

Las tres librerías estrella de recomendadores —**Surprise**, **Implicit**, **LightFM**— no
están instaladas en el laboratorio. En vez de saltearlas, **implementamos su algoritmo central
desde cero con numpy** (SVD-MF con SGD = Surprise; ALS y BPR para implícito = Implicit; híbrido
con features = LightFM) y montamos el benchmark comparativo. El código real de cada librería
queda como referencia. Todo ejecutable, sin internet.

### Setup: dataset sintético + split + NDCG/recall

In [ ]:
import numpy as np, time, pandas as pd
rng = np.random.default_rng(0)
n_users, n_items, n_gen = 60, 40, 5
item_gen = rng.integers(0, n_gen, n_items)
item_feat = np.eye(n_gen)[item_gen]
user_aff = rng.random((n_users, n_gen))
prob = user_aff[:, item_gen]; prob /= prob.sum(1, keepdims=True)

R = np.zeros((n_users, n_items))
for u in range(n_users):
    picks = rng.choice(n_items, size=rng.integers(4, 12), replace=False, p=prob[u])
    R[u, picks] = 1

# held-out 1 positivo por usuario
Rtr = R.copy(); test_pos = {}
for u in range(n_users):
    pos = np.where(R[u] > 0)[0]
    if len(pos) >= 3:
        h = rng.choice(pos); Rtr[u, h] = 0; test_pos[u] = h
users_eval = list(test_pos.keys())

def ndcg_at_k(ranked, relevant, k=10):
    rels = [1.0 if it in relevant else 0.0 for it in ranked[:k]]
    dcg = sum(r/np.log2(i+2) for i, r in enumerate(rels))
    idcg = sum(1.0/np.log2(i+2) for i in range(min(len(relevant), k)))
    return dcg/idcg if idcg else 0.0

def evaluate(score_fn, k=10):
    ndcgs, recalls = [], []
    for u in users_eval:
        s = score_fn(u).copy(); s[Rtr[u] > 0] = -np.inf
        ranked = np.argsort(s)[::-1][:k]
        ndcgs.append(ndcg_at_k(ranked, {test_pos[u]}, k))
        recalls.append(1.0 if test_pos[u] in ranked else 0.0)
    return float(np.mean(ndcgs)), float(np.mean(recalls))

print("usuarios evaluables:", len(users_eval))
print("OK — dataset y métricas listos")

### Ejercicio 1 — Surprise SVD (MF con SGD, RMSE por CV)

Surprise entrena `r̂ = μ + b_u + b_i + p_u·q_i` con SGD y reporta RMSE. Lo implementamos y
hacemos una validación cruzada simple sobre los ratings observados.

In [ ]:
# --- Surprise real (referencia) --------------------------------------------
# from surprise import SVD, Dataset; cross_validate(SVD(), data, measures=['RMSE'], cv=5)
# ---------------------------------------------------------------------------
obs = np.argwhere(R > 0)

def train_mf(train, f=10, lr=0.02, reg=0.05, epochs=40, seed=0):
    rng = np.random.default_rng(seed)
    mu = 1.0
    bu = np.zeros(n_users); bi = np.zeros(n_items)
    P = rng.normal(0, 0.1, (n_users, f)); Q = rng.normal(0, 0.1, (n_items, f))
    for _ in range(epochs):
        rng.shuffle(train)
        for u, i in train:
            e = R[u, i] - (mu + bu[u] + bi[i] + P[u] @ Q[i])
            bu[u] += lr*(e - reg*bu[u]); bi[i] += lr*(e - reg*bi[i])
            P[u] += lr*(e*Q[i] - reg*P[u]); Q[i] += lr*(e*P[u] - reg*Q[i])
    return mu, bu, bi, P, Q

rmses = []
folds = np.array_split(rng.permutation(len(obs)), 5)
for fold in folds:
    test_mask = np.zeros(len(obs), bool); test_mask[fold] = True
    mu, bu, bi, P, Q = train_mf(obs[~test_mask].copy())
    err = [R[u, i] - (mu + bu[u] + bi[i] + P[u] @ Q[i]) for u, i in obs[test_mask]]
    rmses.append(np.sqrt(np.mean(np.square(err))))
print(f"RMSE por fold: {np.round(rmses, 3).tolist()}")
print(f"RMSE medio (5-fold CV): {np.mean(rmses):.3f}")
assert len(rmses) == 5 and np.mean(rmses) < 1.0
print("OK ejercicio 1 — Surprise SVD (MF+SGD) con CV, RMSE reportado")

### Ejercicio 2 — Implicit ALS

`implicit.als.AlternatingLeastSquares`: preferencia binaria + confianza `1+α·r`. Implementado
desde cero; recomendamos y medimos NDCG/recall.

In [ ]:
def implicit_als(Rm, factors=8, alpha=40.0, reg=0.1, iters=12, seed=0):
    rng = np.random.default_rng(seed); nu, ni = Rm.shape
    X = rng.normal(0, 0.01, (nu, factors)); Y = rng.normal(0, 0.01, (ni, factors))
    P = (Rm > 0).astype(float); C = 1.0 + alpha*Rm; eye = reg*np.eye(factors)
    for _ in range(iters):
        YtY = Y.T @ Y
        for u in range(nu):
            Cu = C[u]; X[u] = np.linalg.solve(YtY + (Y.T*(Cu-1))@Y + eye, (Y.T*Cu)@P[u])
        XtX = X.T @ X
        for i in range(ni):
            Ci = C[:, i]; Y[i] = np.linalg.solve(XtX + (X.T*(Ci-1))@X + eye, (X.T*Ci)@P[:, i])
    return X, Y

t0 = time.perf_counter(); Xa, Ya = implicit_als(Rtr); t_als = time.perf_counter() - t0
als_score = lambda u: Xa[u] @ Ya.T
ndcg_als, rec_als = evaluate(als_score)
print(f"ALS  NDCG@10={ndcg_als:.3f} recall@10={rec_als:.3f} train={t_als*1000:.0f}ms")
assert 0 <= ndcg_als <= 1
print("OK ejercicio 2 — Implicit ALS desde cero")

### Ejercicio 3 — Implicit BPR

`implicit.bpr.BayesianPersonalizedRanking`: optimiza el orden par-a-par (item positivo debe
puntuar más que uno negativo). Comparamos NDCG vs ALS.

In [ ]:
def bpr(Rm, factors=8, lr=0.05, reg=0.01, epochs=40, seed=0):
    rng = np.random.default_rng(seed); nu, ni = Rm.shape
    X = rng.normal(0, 0.1, (nu, factors)); Y = rng.normal(0, 0.1, (ni, factors))
    pos = {u: np.where(Rm[u] > 0)[0] for u in range(nu)}
    for _ in range(epochs):
        for u in range(nu):
            if len(pos[u]) == 0: continue
            i = rng.choice(pos[u]); j = rng.integers(0, ni)
            while Rm[u, j] > 0: j = rng.integers(0, ni)
            xuij = X[u] @ (Y[i] - Y[j])
            sig = 1.0 / (1.0 + np.exp(xuij))
            xu = X[u].copy()
            X[u] += lr*(sig*(Y[i]-Y[j]) - reg*X[u])
            Y[i] += lr*(sig*xu - reg*Y[i])
            Y[j] += lr*(-sig*xu - reg*Y[j])
    return X, Y

t0 = time.perf_counter(); Xb, Yb = bpr(Rtr); t_bpr = time.perf_counter() - t0
bpr_score = lambda u: Xb[u] @ Yb.T
ndcg_bpr, rec_bpr = evaluate(bpr_score)
print(f"BPR  NDCG@10={ndcg_bpr:.3f} recall@10={rec_bpr:.3f} train={t_bpr*1000:.0f}ms")
print(f"(ALS NDCG={ndcg_als:.3f})")
assert 0 <= ndcg_bpr <= 1
print("OK ejercicio 3 — Implicit BPR (ranking par-a-par) vs ALS")

### Ejercicio 4 — LightFM (WARP con item features)

LightFM mezcla CF con features de item. Sin la librería, combinamos ALS con la similitud de
features de género (el aporte "hybrid"). Comparamos contra CF puro.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
prof = np.zeros((n_users, n_gen))
for u in range(n_users):
    p = np.where(Rtr[u] > 0)[0]
    if len(p): prof[u] = item_feat[p].mean(0)
feat_sim = cosine_similarity(prof, item_feat)

def norm01(v):
    return (v - v.min()) / (v.max() - v.min() + 1e-9)
lightfm_score = lambda u: 0.5*norm01(Xa[u] @ Ya.T) + 0.5*norm01(feat_sim[u])
t0 = time.perf_counter(); ndcg_lfm, rec_lfm = evaluate(lightfm_score); t_lfm = t_als + (time.perf_counter()-t0)
print(f"LightFM-like NDCG@10={ndcg_lfm:.3f} recall@10={rec_lfm:.3f}")
print(f"(CF puro ALS NDCG={ndcg_als:.3f})")
assert 0 <= ndcg_lfm <= 1
print("OK ejercicio 4 — híbrido con features (concepto LightFM WARP)")

### Ejercicio 5 — Benchmarking de las 4 librerías

Mismo dataset y split para las 4 aproximaciones. Tabla con NDCG@10, recall@10 y tiempo de
entrenamiento — el tipo de comparación que harías con las librerías reales instaladas.

In [ ]:
# Surprise-SVD como recomendador top-N: score = mu+bu+bi+P·Q
mu, bu, bi, P, Q = train_mf(obs.copy())
svd_score = lambda u: mu + bu[u] + bi + Q @ P[u]
ndcg_svd, rec_svd = evaluate(svd_score)

bench = pd.DataFrame([
    {"libreria": "Surprise (SVD)",  "ndcg@10": ndcg_svd, "recall@10": rec_svd, "train_ms": None},
    {"libreria": "Implicit (ALS)",  "ndcg@10": ndcg_als, "recall@10": rec_als, "train_ms": t_als*1000},
    {"libreria": "Implicit (BPR)",  "ndcg@10": ndcg_bpr, "recall@10": rec_bpr, "train_ms": t_bpr*1000},
    {"libreria": "LightFM (hybrid)","ndcg@10": ndcg_lfm, "recall@10": rec_lfm, "train_ms": None},
]).round(3)
print(bench.to_string(index=False))

assert len(bench) == 4
assert bench["ndcg@10"].between(0, 1).all()
print("OK ejercicio 5 — benchmark comparativo de las 4 aproximaciones")